In [1]:
import os
import glob
import scipy.io
import numpy as np
import matplotlib.pyplot as plt

# Path folder
folder_path = r'C:\Users\akip\Desktop\EEGdenoiseNet\Data EEGDenoiseNet'

# Cek semua file di folder
print("Files in folder:")
all_files = os.listdir(folder_path)
for file in all_files:
    print(f"  - {file}")

# Cari file .mat
mat_files = glob.glob(os.path.join(folder_path, "*.mat"))
print(f"\nFile .mat ditemukan: {len(mat_files)}")
for mat_file in mat_files:
    print(f"  - {os.path.basename(mat_file)}")

Files in folder:
  - EEG_all_epochs.mat
  - EEG_all_epochs.npy
  - EMG_all_epochs.mat
  - EMG_all_epochs.npy
  - EOG_all_epochs.mat
  - EOG_all_epochs.npy

File .mat ditemukan: 3
  - EEG_all_epochs.mat
  - EMG_all_epochs.mat
  - EOG_all_epochs.mat


In [3]:
# Jika masih ragu, bisa cek dengan cara ini:
if len(mat_files) > 0:
    mat_path = mat_files[0]
    data = scipy.io.loadmat(mat_path)
    
    # Cari key yang berisi data numerik
    for key in data.keys():
        if not key.startswith('__'):
            value = data[key]
            if isinstance(value, np.ndarray) and value.ndim >= 2:
                print(f"\nData pada key '{key}':")
                print(f"Shape: {value.shape}")
                
                # Tampilkan informasi statistik per channel
                if value.ndim == 2:
                    # Asumsi format (channel, samples)
                    if value.shape[0] < value.shape[1]:
                        n_channels = value.shape[0]
                        print(f"Jumlah channel: {n_channels}")
                        print(f"Panjang sinyal per channel: {value.shape[1]} samples")
                    else:
                        n_channels = value.shape[1]
                        print(f"Jumlah channel: {n_channels}")
                        print(f"Panjang sinyal per channel: {value.shape[0]} samples")


Data pada key 'EEG_all_epochs':
Shape: (4514, 512)
Jumlah channel: 512
Panjang sinyal per channel: 4514 samples

Data pada key 'fs':
Shape: (1, 1)
Jumlah channel: 1
Panjang sinyal per channel: 1 samples


In [6]:
import pyedflib
import os
import numpy as np
import glob

def analyze_edf_file(edf_path):
    """Menganalisis file EDF untuk mengetahui informasi channel"""
    try:
        # Baca file EDF
        with pyedflib.EdfReader(edf_path) as f:
            # Dapatkan informasi header
            n_channels = f.signals_in_file
            channel_labels = f.getSignalLabels()
            n_samples = f.getNSamples()[0]  # samples per channel
            duration = f.getFileDuration()
            sampling_freq = f.getSampleFrequency(0)
            
            print(f"=== ANALISIS FILE: {os.path.basename(edf_path)} ===")
            print(f"Jumlah channel: {n_channels}")
            print(f"Frekuensi sampling: {sampling_freq} Hz")
            print(f"Durasi recording: {duration} detik")
            print(f"Samples per channel: {n_samples}")
            print(f"\nDaftar channel:")
            
            for i, label in enumerate(channel_labels):
                print(f"  {i+1:2d}. {label}")
            
            # Baca data sinyal (opsional, untuk verifikasi)
            signals = np.zeros((n_channels, n_samples))
            for i in range(n_channels):
                signals[i, :] = f.readSignal(i)
            
            print(f"\nBentuk data EEG: {signals.shape}")
            print(f"Format: {signals.shape[0]} channel x {signals.shape[1]} samples")
            
            return signals, channel_labels, sampling_freq
            
    except Exception as e:
        print(f"Error membaca file {edf_path}: {str(e)}")
        return None, None, None

def analyze_edf_event(event_path):
    """Menganalisis file event"""
    try:
        with open(event_path, 'r') as f:
            events = f.readlines()
        
        print(f"\n=== EVENT DATA: {os.path.basename(event_path)} ===")
        print(f"Jumlah event: {len(events)}")
        for i, event in enumerate(events[:10]):  # Tampilkan 10 event pertama
            print(f"  Event {i+1}: {event.strip()}")
        if len(events) > 10:
            print(f"  ... dan {len(events)-10} event lainnya")
            
    except Exception as e:
        print(f"Error membaca event file: {str(e)}")

def analyze_all_edf_files(directory="."):
    """Analisis semua file EDF dalam direktori"""
    # Ganti ke directory yang sesuai
    os.chdir(directory)
    
    edf_files = glob.glob("*.edf")
    edf_files = [f for f in edf_files if not f.endswith('.event')]
    
    print(f"Ditemukan {len(edf_files)} file EDF:")
    
    for edf_file in edf_files:
        print(f"\n{'='*60}")
        # Gunakan fungsi analyze_edf_file, bukan analyze_edf_with_mne
        eeg_data, channels, fs = analyze_edf_file(edf_file)
        
        # Cek file event yang sesuai
        event_file = edf_file + ".event"
        if os.path.exists(event_file):
            analyze_edf_event(event_file)

# Contoh penggunaan untuk single file
if __name__ == "__main__":
    # Path ke dataset Anda
    dataset_path = r"C:\Users\akip\Desktop\EEGdenoiseNet\eeg-motor-movementimagery-dataset-1.0.0"
    
    # Coba analisis satu file dulu
    test_file = os.path.join(dataset_path, "S001", "S001R01.edf")
    
    if os.path.exists(test_file):
        print("Menganalisis file tunggal...")
        eeg_data, channels, fs = analyze_edf_file(test_file)
        
        if eeg_data is not None:
            print(f"\n=== INFORMASI TAMBAHAN ===")
            print(f"Range data channel 1: {eeg_data[0].min():.2f} hingga {eeg_data[0].max():.2f}")
            print(f"5 sample pertama channel 1: {eeg_data[0][:5]}")
    else:
        print(f"File {test_file} tidak ditemukan!")
        print("Mencari struktur folder...")
        
        # Analisis struktur folder
        analyze_dataset_structure(dataset_path)

def analyze_dataset_structure(directory):
    """Analisis struktur dataset"""
    print(f"\nAnalisis struktur dataset di: {directory}")
    
    # Cari semua folder subject
    subjects = [d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d)) and d.startswith('S')]
    print(f"Ditemukan {len(subjects)} subjects: {subjects}")
    
    if subjects:
        # Analisis subject pertama
        first_subject = os.path.join(directory, subjects[0])
        print(f"\nAnalisis files di {first_subject}:")
        
        edf_files = glob.glob(os.path.join(first_subject, "*.edf"))
        edf_files = [f for f in edf_files if not f.endswith('.event')]
        
        for edf_file in edf_files[:2]:  # Analisis 2 file pertama saja
            print(f"\n{'='*60}")
            eeg_data, channels, fs = analyze_edf_file(edf_file)
            
            # Cek file event
            event_file = edf_file + ".event"
            if os.path.exists(event_file):
                analyze_edf_event(event_file)



Menganalisis file tunggal...
=== ANALISIS FILE: S001R01.edf ===
Jumlah channel: 64
Frekuensi sampling: 160.0 Hz
Durasi recording: 61.0 detik
Samples per channel: 9760

Daftar channel:
   1. Fc5.
   2. Fc3.
   3. Fc1.
   4. Fcz.
   5. Fc2.
   6. Fc4.
   7. Fc6.
   8. C5..
   9. C3..
  10. C1..
  11. Cz..
  12. C2..
  13. C4..
  14. C6..
  15. Cp5.
  16. Cp3.
  17. Cp1.
  18. Cpz.
  19. Cp2.
  20. Cp4.
  21. Cp6.
  22. Fp1.
  23. Fpz.
  24. Fp2.
  25. Af7.
  26. Af3.
  27. Afz.
  28. Af4.
  29. Af8.
  30. F7..
  31. F5..
  32. F3..
  33. F1..
  34. Fz..
  35. F2..
  36. F4..
  37. F6..
  38. F8..
  39. Ft7.
  40. Ft8.
  41. T7..
  42. T8..
  43. T9..
  44. T10.
  45. Tp7.
  46. Tp8.
  47. P7..
  48. P5..
  49. P3..
  50. P1..
  51. Pz..
  52. P2..
  53. P4..
  54. P6..
  55. P8..
  56. Po7.
  57. Po3.
  58. Poz.
  59. Po4.
  60. Po8.
  61. O1..
  62. Oz..
  63. O2..
  64. Iz..

Bentuk data EEG: (64, 9760)
Format: 64 channel x 9760 samples

=== INFORMASI TAMBAHAN ===
Range data channel 1:

In [7]:
# Jalankan analisis
if __name__ == "__main__":
    dataset_path = r"C:\Users\akip\Desktop\EEGdenoiseNet\eeg-motor-movementimagery-dataset-1.0.0"
    
    if os.path.exists(dataset_path):
        print(f"Dataset ditemukan di: {dataset_path}")
        analyze_dataset_structure(dataset_path)
    else:
        print(f"Path {dataset_path} tidak ditemukan!")
        # Coba path alternatif
        current_dir = os.getcwd()
        print(f"Current directory: {current_dir}")
        print(f"Files in current directory: {os.listdir('.')}")

Dataset ditemukan di: C:\Users\akip\Desktop\EEGdenoiseNet\eeg-motor-movementimagery-dataset-1.0.0

Analisis struktur dataset di: C:\Users\akip\Desktop\EEGdenoiseNet\eeg-motor-movementimagery-dataset-1.0.0
Ditemukan 17 subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010', 'S011', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017']

Analisis files di C:\Users\akip\Desktop\EEGdenoiseNet\eeg-motor-movementimagery-dataset-1.0.0\S001:

=== ANALISIS FILE: S001R01.edf ===
Jumlah channel: 64
Frekuensi sampling: 160.0 Hz
Durasi recording: 61.0 detik
Samples per channel: 9760

Daftar channel:
   1. Fc5.
   2. Fc3.
   3. Fc1.
   4. Fcz.
   5. Fc2.
   6. Fc4.
   7. Fc6.
   8. C5..
   9. C3..
  10. C1..
  11. Cz..
  12. C2..
  13. C4..
  14. C6..
  15. Cp5.
  16. Cp3.
  17. Cp1.
  18. Cpz.
  19. Cp2.
  20. Cp4.
  21. Cp6.
  22. Fp1.
  23. Fpz.
  24. Fp2.
  25. Af7.
  26. Af3.
  27. Afz.
  28. Af4.
  29. Af8.
  30. F7..
  31. F5..
  32. F3..
  33. F1..
  34. Fz..
  35